# The common API shape: messages, stop reason, usage

> 📓 *Companion to* **Modern Agentic AI Engineer** *· Ch 11 §11.1–11.3 · type: walkthrough*

**One-line promise:** by the end you can read any provider's response correctly — branch on the *stop reason*, log *usage*, walk a list of typed content blocks, and attach an image — and see that Anthropic and OpenAI differ mostly in dialect, not anatomy.

## 🧠 Why this matters

Strip away the branding and every chat API is the same animal: you POST a model id, a list of role-tagged messages, and some generation params (plus optional tools); you get back **content**, a **stop reason** (why generation ended), and token **usage**. Teams that treat the call casually ship truncated answers as if they were complete, lose all cost visibility, and rewrite forty call sites the day they want to swap providers.

Get the response *anatomy* into your hands now — once you can read `content`, `stop_reason`, and `usage` on reflex, tool calling and multimodal are just more of the same shape. This notebook runs free and offline: `MOCK=1` returns canned response objects that mirror the real SDKs.

## Objectives & prereqs

**By the end you can:**
- Make a minimal `messages.create(...)` call and read `content[0].text`, `stop_reason`, `usage`.
- Branch correctly on the stop reason — especially silent `max_tokens` truncation.
- Map the Anthropic shape onto the OpenAI dialect (system-in-list, `finish_reason`, `choices`).
- Build an image+text content list and reason about images-are-tokens.

**Prereqs:** Chapters 8–10 and the Ch 10 notebooks (`learn/part-03-llm-substrate/10-*`). No API key needed in mock mode.

In [ ]:
# --- Setup: imports, env, and the MOCK switch ---------------------------------
# Packages used here are stdlib + `python-dotenv` from requirements.txt only.
import os
import base64
import random
from dataclasses import dataclass, field
from pathlib import Path

try:
    from dotenv import load_dotenv
    load_dotenv()  # reads a git-ignored .env if present; no-op otherwise
except ImportError:
    pass  # dotenv is optional in mock mode

# MOCK=1 (default) => canned, deterministic responses: free & offline, no key.
# MOCK=0 => hit the live Anthropic/OpenAI APIs (needs a key; ~2 short calls,
#          one with an image whose tokens are flagged in usage).
MOCK = os.getenv("COMPANION_MOCK", "1") == "1"
random.seed(11)  # any stochastic helper below is reproducible

DATA = Path("data")
print(f"MOCK mode: {MOCK}  (set COMPANION_MOCK=0 in .env to hit live APIs)")
if not MOCK and not os.getenv("ANTHROPIC_API_KEY"):
    raise SystemExit(
        "MOCK=0 but ANTHROPIC_API_KEY is unset — add it to .env or stay in mock mode."
    )

### The mock layer

We model just enough of the SDK's response object to be faithful: an Anthropic message is a list of **typed content blocks**, a `stop_reason`, and a `usage` meter. The mock `claude_create(...)` returns one of these shaped objects so every cell below runs identically with or without a key. In live mode (`MOCK=0`) the same call goes to `anthropic.Anthropic().messages.create(...)`.

In [ ]:
# A faithful-enough stand-in for the Anthropic response object.
@dataclass
class Block:
    type: str
    text: str | None = None
    # tool_use blocks carry id/name/input; image blocks carry source (omitted here)
    id: str | None = None
    name: str | None = None
    input: dict | None = None


@dataclass
class Usage:
    input_tokens: int = 0
    output_tokens: int = 0


@dataclass
class Message:
    content: list = field(default_factory=list)
    stop_reason: str = "end_turn"   # end_turn | max_tokens | tool_use | stop_sequence
    usage: Usage = field(default_factory=Usage)
    model: str = "claude-sonnet-4-6"


def claude_create(*, model, max_tokens, system=None, messages, tools=None, _canned):
    """Mock of claude.messages.create(...). `_canned` selects the scripted reply.
    In live mode we delete `_canned` and call the real SDK instead."""
    if not MOCK:
        import anthropic  # only imported on the live path
        client = anthropic.Anthropic()
        kwargs = dict(model=model, max_tokens=max_tokens, messages=messages)
        if system is not None:
            kwargs["system"] = system
        if tools is not None:
            kwargs["tools"] = tools
        return client.messages.create(**kwargs)
    return _canned()

## 1 · A minimal call, and the three fields that matter

Anthropic takes the system prompt as a **top-level** `system` parameter and requires `max_tokens` explicitly. The reply is a `Message`; the three fields you read on *every* call are `content`, `stop_reason`, and `usage`.

In [ ]:
def _canned_support_answer():
    return Message(
        content=[Block(type="text",
            text="Your order ORD-9912 shipped Tuesday and is out for delivery today.")],
        stop_reason="end_turn",
        usage=Usage(input_tokens=27, output_tokens=18),
        model="claude-sonnet-4-6",
    )

msg = claude_create(
    model="claude-sonnet-4-6",
    max_tokens=1024,
    system="You are the support agent for Acme.",   # top-level parameter
    messages=[{"role": "user", "content": "Where is my order?"}],
    _canned=_canned_support_answer,
)

print("text       :", msg.content[0].text)
print("stop_reason:", msg.stop_reason)
print("usage      :", msg.usage)

**What you just saw.** `content` is a *list* (here, one text block) — not a bare string. `stop_reason="end_turn"` means the model finished naturally. `usage` is your per-call meter: log it on every call, because cost visibility starts here (§11.5).

## 2 · The OpenAI dialect, side by side

Same anatomy, different field names. OpenAI puts the system (or "developer") message **in the list**, returns `choices`, and calls the stop reason `finish_reason`. The chat shape has become a de-facto lingua franca — most open-model servers (vLLM, Ollama …) expose OpenAI-compatible endpoints, so code written against it can often point at self-hosted models unchanged.

In [ ]:
# Mock of the OpenAI chat-completions shape (choices[].message / finish_reason).
@dataclass
class OAIMessage:
    content: str


@dataclass
class OAIChoice:
    message: OAIMessage
    finish_reason: str


@dataclass
class OAIResponse:
    choices: list


def oai_create(*, model, messages, _canned):
    if not MOCK:
        from openai import OpenAI
        return OpenAI().chat.completions.create(model=model, messages=messages)
    return _canned()


resp = oai_create(
    model="gpt-5",
    messages=[
        {"role": "system", "content": "You are the support agent for Acme."},  # system IN the list
        {"role": "user", "content": "Where is my order?"},
    ],
    _canned=lambda: OAIResponse(choices=[OAIChoice(
        message=OAIMessage(content="Your order ORD-9912 is out for delivery today."),
        finish_reason="stop")]),
)
print("text         :", resp.choices[0].message.content)
print("finish_reason:", resp.choices[0].finish_reason)

**Translation table.** Anthropic `content[0].text` ↔ OpenAI `choices[0].message.content`; Anthropic `stop_reason` ↔ OpenAI `finish_reason`; Anthropic top-level `system=` ↔ OpenAI system message in the list. Anatomy identical; dialect different. (Anthropic content stays a *list of blocks* — that pays off in the next sections.)

## 3 · 🔮 Predict: what stop reason when output is cut off?

We send a request with a **tiny** `max_tokens` so the model can't finish. Before running the next cell, predict:

1. What will `stop_reason` be?
2. If your code only does `print(msg.content[0].text)` and ignores the stop reason, what does the user receive?

Write your guess down, then run it.

In [ ]:
def _canned_truncated():
    return Message(
        content=[Block(type="text",
            text="Your order ORD-9912 shipped Tuesday and is currently in transit between the")],
        stop_reason="max_tokens",   # <-- the tell
        usage=Usage(input_tokens=27, output_tokens=16),
    )

msg = claude_create(
    model="claude-sonnet-4-6",
    max_tokens=16,  # deliberately too small to finish the sentence
    system="You are the support agent for Acme.",
    messages=[{"role": "user", "content": "Give me a full status update on ORD-9912."}],
    _canned=_canned_truncated,
)

print("raw text   :", repr(msg.content[0].text))
print("stop_reason:", msg.stop_reason)

**What you just saw.** The text *looks* like a normal answer — it just stops mid-sentence — and nothing raises. The only signal that it's incomplete is `stop_reason == "max_tokens"`. **Correct code branches on the stop reason** instead of trusting the text:

In [ ]:
def handle(msg):
    """Branch on stop_reason instead of trusting the text blindly."""
    if msg.stop_reason == "end_turn":
        return msg.content[0].text
    if msg.stop_reason == "max_tokens":
        # Truncated. Raise max_tokens, or continue the turn — don't ship it as final.
        raise ValueError("Output truncated at max_tokens — raise the cap or continue generation.")
    if msg.stop_reason == "tool_use":
        return "<model requested a tool — execute it and loop>"  # see section 4
    if msg.stop_reason == "stop_sequence":
        return msg.content[0].text  # hit a configured stop string
    raise ValueError(f"Unhandled stop_reason: {msg.stop_reason}")


try:
    handle(msg)
except ValueError as e:
    print("caught:", e)

> ⚠️ **Pitfall — shipping truncated output as if it were complete.** Ignoring `stop_reason` is the single most common API-level bug: the answer reads fine, passes your eyeball test, and is *wrong because it's cut off*. A support agent that confidently emails half a refund policy is worse than one that errors. Always branch on the stop reason.

## 4 · Content is a list of *typed blocks*

On a tool-use turn the Anthropic `content` list mixes block types: the model may emit a `text` block *and* a `tool_use` block. This typed-list shape is exactly why the same anatomy scales to agents and multimodal — you iterate blocks and branch on `block.type` rather than parsing one string. (Chapter 12 builds the full tool loop; here we just *read* the shape.)

In [ ]:
def _canned_tool_turn():
    return Message(
        content=[
            Block(type="text", text="Let me look that order up."),
            Block(type="tool_use", id="toolu_01", name="get_order_status",
                  input={"order_id": "ORD-9912"}),
        ],
        stop_reason="tool_use",
        usage=Usage(input_tokens=88, output_tokens=31),
    )

msg = claude_create(
    model="claude-sonnet-4-6", max_tokens=1024,
    system="You are the support agent for Acme.",
    messages=[{"role": "user", "content": "Where is order ORD-9912?"}],
    tools=[{"name": "get_order_status", "description": "Look up order status by id.",
            "input_schema": {"type": "object",
                             "properties": {"order_id": {"type": "string"}},
                             "required": ["order_id"]}}],
    _canned=_canned_tool_turn,
)

print("stop_reason:", msg.stop_reason, "\n")
for block in msg.content:
    if block.type == "text":
        print("[text]    ", block.text)
    elif block.type == "tool_use":
        print("[tool_use]", block.name, block.input, "(id:", block.id, ")")

**What you just saw.** `stop_reason == "tool_use"` is the signal to *act*: you execute `get_order_status(order_id="ORD-9912")` yourself, append a `tool_result` to the conversation, and call the API again. That `while` loop is, minimally, an agent — Chapter 12 takes it from here. OpenAI's dialect is the same dance with different names (`tool_calls` on the message, results returned in `tool`-role messages).

## 5 · Multimodal: an image+text content list (§11.3)

Vision uses the same typed-block shape: you add an `image` block alongside `text`. We load the tiny committed `data/dashboard.png` (a 96×48 "dashboard" with three service tiles, one red) and base64-encode it exactly as you would a real screenshot.

In [ ]:
img_path = DATA / "dashboard.png"
png_b64 = base64.standard_b64encode(img_path.read_bytes()).decode()
print(f"loaded {img_path} — {img_path.stat().st_size} bytes, {len(png_b64)} base64 chars")

content = [
    {"type": "image", "source": {"type": "base64",
                                 "media_type": "image/png", "data": png_b64}},
    {"type": "text", "text": "Which service on this dashboard is unhealthy, and why?"},
]


def _canned_vision():
    # Image tokens are estimated, not real — but flagged so the cost lesson lands.
    return Message(
        content=[Block(type="text",
            text="The third (right-most) service tile is red, indicating it is unhealthy; "
                 "the other two are green.")],
        stop_reason="end_turn",
        usage=Usage(input_tokens=1190, output_tokens=24),  # ~1000+ of those are the image
    )

msg = claude_create(
    model="claude-sonnet-4-6", max_tokens=1024,
    messages=[{"role": "user", "content": content}],
    _canned=_canned_vision,
)
print("\nanswer:", msg.content[0].text)
print("usage :", msg.usage, "  <- note the inflated input_tokens: images are tokens too")

**What you just saw.** The vision answer is just another `text` block — the *input* changed, not the response anatomy. But look at `input_tokens`: a single hi-res image runs ~1000+ tokens. A screenshot-heavy agent loop inflates context, cost, and time-to-first-token fast. **Downscale images to the resolution the task needs** (ImageMagick / Pillow in your ingestion path) and **don't re-send static images on every loop iteration.**

## 🎯 Senior lens

You now have the three response fields and the typed-block shape in hand — so resist the urge to sprinkle `claude.messages.create(...)` across forty files. **Route every model call through one internal client of your own.** Stop-reason branching, usage logging, image hygiene, retries, timeouts, caching, and model selection then live in *one* place instead of being re-derived (and forgotten) per call site. Swapping providers becomes a config change, not a refactor. That single-door `LLMClient` is the chapter's 🔧 Build — you assemble it in **notebook 11-03**. Everything here is the anatomy it standardizes.

## Recap

- Every chat API shares one anatomy: model id + role-tagged messages + params (+ tools) → **content**, a **stop reason**, and token **usage**.
- Anthropic vs OpenAI differ in *dialect* (`system=` vs system-in-list; `stop_reason` vs `finish_reason`; `content` blocks vs a string), not anatomy.
- **Branch on `stop_reason`** — `max_tokens` is silent truncation; `tool_use` means *act*.
- Anthropic `content` is a **list of typed blocks** (text / tool_use / image) — the shape that scales to agents and multimodal.
- Images are tokens too (~1000+ each): downscale to the task and never re-send static images per loop.
- `usage` is your per-call cost meter — log it on every call.

## Exercises

Each exercise *changes* something and asks you to 🔮 predict the result first.

1. **Add a `tool_result` turn.** Take the section-4 tool-use message, append `{"role": "assistant", "content": msg.content}` plus a user message carrying a `tool_result` block, and write the (mock) follow-up call that produces the final answer. Predict its `stop_reason`.
2. **Continue a truncated turn.** Modify `handle()` so that on `max_tokens` it *continues* generation (append the partial text as an assistant turn and call again) instead of raising. What would you cap to avoid an infinite continue-loop?
3. **Port section 1 to the OpenAI dialect.** Rewrite the minimal call using `oai_create`, reading `choices[0].message.content` and `finish_reason`. Which fields had to move?
4. **Measure image cost.** Re-run the vision cell with a second, larger fake image (bump the mock `input_tokens`). At your provider's input price, what does re-sending that image on every one of 20 agent loops cost vs sending it once?

In [ ]:
# Exercise 1 — your code here

In [ ]:
# Exercise 2 — your code here

In [ ]:
# Exercise 3 — your code here

In [ ]:
# Exercise 4 — your code here

## Next

- ➡️ **Next notebook:** [`11-02-resilience-retries-and-idempotency.ipynb`](./11-02-resilience-retries-and-idempotency.ipynb) — the API fails in categories; build retry logic that retries only what's retryable and survives an ambiguous timeout.
- 🔧 **Builds toward:** the single-door `LLMClient` in [`11-03-caching-batching-and-the-llm-client.ipynb`](./11-03-caching-batching-and-the-llm-client.ipynb).
- 📘 **Blueprint:** the production wrapper at [`blueprints/llm-gateway/`](../../../blueprints/llm-gateway/).
- 🏗️ **Capstone:** advances the platform's `llm/` package (checkpoint `ch11-llm-client`).
- See the book **§11.1–§11.3** for the prose and full code.